<a href="https://colab.research.google.com/github/liqq1024/GenAI_LLMs_2026Fall/blob/main/assignments/Homeweek1/Homework1_Tokenization_Embeddings_RNN_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 1 : Tokenization, Embeddings, and Sequence Modeling with PyTorch

This instructor solution follows the published Homework 1 requirements. Run the notebook from top to bottom. Because the dataset is intentionally small, exact validation metrics and generated text may vary slightly across environments.

## Setup

Google Colab already includes PyTorch. The next cell installs the remaining libraries used for tokenization and PCA.

In [ ]:
!pip -q install transformers tokenizers datasets scikit-learn

import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from transformers import AutoTokenizer
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## Part 1: Tokenization with GPT-2 and BERT

In [ ]:
text = "Large Language Models are changing AI."

gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

## Answer the questions 1-4 in part 1:

## Part 2: Create a Dataset for Next-Word Prediction

The custom word-level vocabulary below is separate from the pre-trained GPT-2 and BERT vocabularies used in Part 1.

In [ ]:
sentences = [
    "healthy plants need sunlight and water",
    "disease symptoms can appear on leaves",
    "early detection helps protect crops",
    "machine learning can identify plant diseases",
    "farmers use images to monitor plant health",
    "deep learning models learn patterns from data",
    "sensors can support precision agriculture",
    "timely treatment can reduce crop damage",
    "leaf spots may indicate fungal infection",
    "data quality affects model performance",
]


### Answer the questions 1-4 in part 2

## Shared training and evaluation helpers

In [ ]:
def evaluate_model(model, data_loader, criterion):
    model.eval()
    total_loss = total_correct = total_examples = 0
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            total_loss += loss.item() * X_batch.size(0)
            total_correct += (logits.argmax(dim=1) == y_batch).sum().item()
            total_examples += y_batch.size(0)
    return total_loss / total_examples, total_correct / total_examples


def train_model(model, train_loader, val_loader, epochs=100, learning_rate=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(epochs):
        model.train()
        total_loss = total_correct = total_examples = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * X_batch.size(0)
            total_correct += (logits.argmax(dim=1) == y_batch).sum().item()
            total_examples += y_batch.size(0)

        train_loss = total_loss / total_examples
        train_acc = total_correct / total_examples
        val_loss, val_acc = evaluate_model(model, val_loader, criterion)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        if epoch == 0 or (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch + 1:3d}/{epochs} | train loss {train_loss:.3f}, acc {train_acc:.1%} | val loss {val_loss:.3f}, acc {val_acc:.1%}")
    return history


def plot_history(history, title):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, history["train_loss"], label="Training")
    axes[0].plot(epochs, history["val_loss"], label="Validation")
    axes[0].set(title=f"{title}: Loss", xlabel="Epoch", ylabel="Cross-entropy loss")
    axes[1].plot(epochs, history["train_acc"], label="Training")
    axes[1].plot(epochs, history["val_acc"], label="Validation")
    axes[1].set(title=f"{title}: Accuracy", xlabel="Epoch", ylabel="Accuracy")
    for ax in axes:
        ax.legend()
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## Part 3: Train a SimpleRNN Model

### Answer the question 1-3 in part 3

## Part 4: Train an LSTM Model

## Part 5: Visualize and Analyze Embeddings

### Answer the questions 1-4 in Part 5


## Part 6: Compare the Models

In [ ]:
print("| Criterion | SimpleRNN | LSTM |")
print("|---|---:|---:|")
print(f"| Final training accuracy | {rnn_history['train_acc'][-1]:.2%} | {lstm_history['train_acc'][-1]:.2%} |")
print(f"| Final validation accuracy | {rnn_history['val_acc'][-1]:.2%} | {lstm_history['val_acc'][-1]:.2%} |")

### Answer the questions 1-5 in Part 6

## Part 7: Generate Text

## Part 8: Reflection

